# 01 Reconcile: Kaggle `ipl_cricket` vs Cricsheet `crickql.raw`

Before anything downstream trusts the new pipeline, compare it with the load I already had.
Three levels, each one narrower than the last:

1. **Season:** matches, deliveries and runs per season
2. **Match:** which match IDs exist on only one side
3. **Delivery:** for matches on both sides, do delivery counts and runs agree?

Every gap gets an explanation in the findings cell at the bottom. A gap with no explanation is a bug until proven otherwise.

Kaggle columns used: `matches(id, season, date)`, `deliveries(match_id, inning, total_runs)`.

In [ ]:
import os

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

load_dotenv("../.env")
BASE = dict(host=os.getenv("PG_HOST", "localhost"), port=int(os.getenv("PG_PORT", "5432")),
            username=os.getenv("PG_USER") or None, password=os.getenv("PG_PASSWORD") or None)
KAGGLE_DB = "ipl_cricket"
CRICKQL_DB = os.getenv("PG_DB", "crickql")


ENGINES = {}


def q(db, sql):
    if db not in ENGINES:
        ENGINES[db] = create_engine(URL.create("postgresql+psycopg2", database=db, **BASE))
    with ENGINES[db].connect() as conn:
        return pd.read_sql(text(sql), conn)

## 1. Season level

In [ ]:
kaggle_season = q(KAGGLE_DB, """
    SELECT m.season::text AS season,
           count(DISTINCT m.id)  AS k_matches,
           count(d.match_id)     AS k_deliveries,
           sum(d.total_runs)     AS k_runs
    FROM matches m LEFT JOIN deliveries d ON d.match_id = m.id
    GROUP BY 1
""")
cricsheet_season = q(CRICKQL_DB, """
    SELECT m.season_raw AS season, min(m.season_year) AS season_year,
           count(DISTINCT m.match_id) AS c_matches,
           count(d.match_id)          AS c_deliveries,
           sum(d.runs_total)          AS c_runs
    FROM raw.matches m LEFT JOIN raw.deliveries d USING (match_id)
    GROUP BY 1
""")
season = cricsheet_season.merge(kaggle_season, on="season", how="outer").sort_values("season")
for col in ("matches", "deliveries", "runs"):
    season[f"diff_{col}"] = season[f"c_{col}"].fillna(0) - season[f"k_{col}"].fillna(0)
season

In [ ]:
season_gaps = season[(season[["diff_matches", "diff_deliveries", "diff_runs"]] != 0).any(axis=1)]
season_gaps

## 2. Match level

In [ ]:
k_ids = set(q(KAGGLE_DB, "SELECT id FROM matches")["id"])
c = q(CRICKQL_DB, "SELECT match_id, season_raw, match_date, result_type FROM raw.matches")
c_ids = set(c["match_id"])
only_cricsheet = c[c["match_id"].isin(c_ids - k_ids)]
only_kaggle = sorted(k_ids - c_ids)
print(f"both: {len(k_ids & c_ids)}  only Cricsheet: {len(only_cricsheet)}  "
      f"only Kaggle: {len(only_kaggle)}")
only_cricsheet.groupby("season_raw").size().rename("matches_only_in_cricsheet")

In [ ]:
# Kaggle matches Cricsheet does not have (should be empty; anything here needs a reason)
only_kaggle[:20]

## 3. Delivery level, matches on both sides

In [ ]:
k_match = q(KAGGLE_DB, """
    SELECT match_id, count(*) AS k_deliveries, sum(total_runs) AS k_runs,
           count(*) FILTER (WHERE inning > 2) AS k_super_over_balls
    FROM deliveries GROUP BY 1
""")
c_match = q(CRICKQL_DB, """
    SELECT match_id, count(*) AS c_deliveries, sum(runs_total) AS c_runs,
           count(*) FILTER (WHERE is_super_over) AS c_super_over_balls
    FROM raw.deliveries GROUP BY 1
""")
both = c_match.merge(k_match, on="match_id", how="inner")
mismatch = both[(both.c_deliveries != both.k_deliveries) | (both.c_runs != both.k_runs)]
print(f"matches compared: {len(both)}, mismatched: {len(mismatch)}")
mismatch.head(20)

In [ ]:
# Numbers for the findings cell
total_c = q(CRICKQL_DB, "SELECT (SELECT count(*) FROM raw.matches) m, "
                        "(SELECT count(*) FROM raw.deliveries) d")
total_k = q(KAGGLE_DB, "SELECT (SELECT count(*) FROM matches) m, "
                       "(SELECT count(*) FROM deliveries) d")
print("Cricsheet:", total_c.iloc[0].to_dict())
print("Kaggle:   ", total_k.iloc[0].to_dict())
print("Seasons with gaps:", season_gaps["season"].tolist())
print("Super-over balls, Kaggle vs Cricsheet:",
      int(both.k_super_over_balls.sum()), int(both.c_super_over_balls.sum()))

## Findings

*Write this after running every cell above. One bullet per gap, each with a cause and evidence. Use only numbers the cells printed.*

- **Overall:**
- **Seasons only in Cricsheet:**
- **Matches only in Kaggle:**
- **Delivery-level mismatches:**
- **Super overs:**
- **Verdict:** can the Cricsheet pipeline replace the Kaggle load? Yes or no, and why.